<a href="https://colab.research.google.com/github/teoteoh/sctec_aulas/blob/main/Miniprojeto_TeodoraCosta_Analise_de_Dados_TI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Importando as bibliotecas relevantes:

In [23]:
import csv
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re

Carregando CSV e fazendo as primeiras
inspeções:

In [24]:
df = pd.read_csv('Base Varejo.csv', sep=';')

display(df.head())


,DATA,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,01/02/2019,1000,534,M,4,1,C,67,BEBIDAS,REFRIGERANTE GUARANA,NaN,NaN,NaN,NaN
1,01/02/2019,1000,534,M,4,1,C,70,BEBIDAS,REFRIGERANTE OUTROS,NaN,NaN,NaN,NaN
2,01/02/2019,1000,534,M,4,1,C,178,HIGIENE,LENCO UMEDECIDO,NaN,NaN,NaN,NaN
3,01/02/2019,1000,534,M,4,1,C,4,ALIMENTOS,ABACAXI,NaN,NaN,NaN,NaN
4,01/02/2019,1000,534,M,4,1,C,175,LIMPEZA,LIMPADOR MULTIUSO,NaN,NaN,NaN,NaN


In [25]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 830000 entries, 0 to 829999
Data columns (total 14 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   DATA         830000 non-null  object 
 1   CO_ID        830000 non-null  int64  
 2   CL_ID        830000 non-null  int64  
 3   CL_GENERO    830000 non-null  object 
 4   CL_EC        830000 non-null  int64  
 5   CL_FHL       830000 non-null  int64  
 6   CL_SEG       830000 non-null  object 
 7   PR_ID        830000 non-null  int64  
 8   PR_CAT       830000 non-null  object 
 9   PR_NOME      830000 non-null  object 
 10  Unnamed: 10  0 non-null       float64
 11  Unnamed: 11  0 non-null       float64
 12  Unnamed: 12  0 non-null       float64
 13  Unnamed: 13  0 non-null       float64
dtypes: float64(4), int64(5), object(5)
memory usage: 88.7+ MB
None


Há colunas sem rótulo e com valores vazios. Fazemos a remoção das colunas vazias para manter a base de dados mais limpa e concisa:

In [26]:
df = df.dropna(axis=1, how='all')

Checamos para ver se há linhas duplicadas:

In [27]:
total_duplicadas = df.duplicated().sum()


if total_duplicadas > 0:
    print(f"Linhas duplicadas: {total_duplicadas}")
else:
    print("\nNão há duplicatas.")

Linhas duplicadas: 96553


Agora removemos duplicatas e checamos o resultado:

In [28]:
df = df.drop_duplicates()

print(df.info())

<class 'pandas.core.frame.DataFrame'>
Index: 733447 entries, 0 to 829999
Data columns (total 10 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   DATA       733447 non-null  object
 1   CO_ID      733447 non-null  int64 
 2   CL_ID      733447 non-null  int64 
 3   CL_GENERO  733447 non-null  object
 4   CL_EC      733447 non-null  int64 
 5   CL_FHL     733447 non-null  int64 
 6   CL_SEG     733447 non-null  object
 7   PR_ID      733447 non-null  int64 
 8   PR_CAT     733447 non-null  object
 9   PR_NOME    733447 non-null  object
dtypes: int64(5), object(5)
memory usage: 61.6+ MB
None


Transformando a coluna DATA em DataTime:

In [29]:
df['DATA'] = pd.to_datetime(df['DATA'], format='%d/%m/%Y')

display(df.head())
print(df['DATA'].dtype)

,DATA,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME
0,2019-02-01,1000,534,M,4,1,C,67,BEBIDAS,REFRIGERANTE GUARANA
1,2019-02-01,1000,534,M,4,1,C,70,BEBIDAS,REFRIGERANTE OUTROS
2,2019-02-01,1000,534,M,4,1,C,178,HIGIENE,LENCO UMEDECIDO
3,2019-02-01,1000,534,M,4,1,C,4,ALIMENTOS,ABACAXI
4,2019-02-01,1000,534,M,4,1,C,175,LIMPEZA,LIMPADOR MULTIUSO


datetime64[ns]


Agora verificamos se há categorias vazias:

In [30]:
print("NaN por coluna:")
print(df.isna().sum())

print("Strings por coluna:")
for col in df.select_dtypes(include=['object']).columns:
    vazios = (df[col].astype(str).str.strip() == '').sum()
    if vazios > 0:
        print(f"{col}: {vazios} registros vazios.")
    else:
        print(f"{col}: Nenhum registro vazio.")

NaN por coluna:
DATA         0
CO_ID        0
CL_ID        0
CL_GENERO    0
CL_EC        0
CL_FHL       0
CL_SEG       0
PR_ID        0
PR_CAT       0
PR_NOME      0
dtype: int64
Strings por coluna:
CL_GENERO: Nenhum registro vazio.
CL_SEG: Nenhum registro vazio.
PR_CAT: Nenhum registro vazio.
PR_NOME: Nenhum registro vazio.


Substituimos os NaN por "Sem Categoria" e checamos o resultado:

In [31]:
df = df.fillna('Sem Categoria')

print(f"Valores vazios: {df.isna().sum().sum()}")

display(df.tail())

Valores vazios: 0


,DATA,CO_ID,CL_ID,CL_GENERO,CL_EC,CL_FHL,CL_SEG,PR_ID,PR_CAT,PR_NOME
829991,2022-08-19,919822,155,F,2,0,B,86,HIGIENE,PRESERVATIVO
829993,2022-08-19,919822,155,F,2,0,B,62,ALIMENTOS,SNACKS
829994,2022-08-19,919822,155,F,2,0,B,11,ALIMENTOS,AZEITE
829998,2022-08-19,919822,155,F,2,0,B,214,ALIMENTOS,CEBOLA
829999,2022-08-19,919822,155,F,2,0,B,59,ALIMENTOS,SALGADINHO


Estatísticas básicas relacionadas ao número de filhos dos clientes:

In [32]:
stats = df['CL_FHL'].describe()
mode_val = df['CL_FHL'].mode()[0]
median_val = df['CL_FHL'].median()

print("Estatísticas Descritivas Referentes ao Número de Filhos:")
print(f"Contagem:        {stats['count']}")
print(f"Média:           {stats['mean']:.2f}")
print(f"Mediana:         {median_val}")
print(f"Moda:            {mode_val}")
print(f"Desvio Padrão:   {stats['std']:.2f}")
print(f"Mínimo:          {stats['min']}")
print(f"Máximo:          {stats['max']}")
print(f"Quartil 25%:     {stats['25%']}")
print(f"Quartil 50%:     {stats['50%']}")
print(f"Quartil 75%:     {stats['75%']}")

Estatísticas Descritivas Referentes ao Número de Filhos:
Contagem:        733447.0
Média:           1.15
Mediana:         0.0
Moda:            0
Desvio Padrão:   1.42
Mínimo:          0.0
Máximo:          4.0
Quartil 25%:     0.0
Quartil 50%:     0.0
Quartil 75%:     2.0
